# 07. cuDF Import/Export and GPU Acceleration
## 📚 Learning Objectives

By completing this notebook, you will:
- Import and export data in different formats using cuDF functions
- Use GPU acceleration from cuDF to process data faster
- Compare cuDF performance with Pandas
- Optimize data processing using GPU acceleration

## 🔗 Prerequisites

- ✅ Examples 01–06 (data loading through EDA)
- ✅ Understanding of Pandas basics
- ✅ CUDA-capable GPU (optional, but recommended)

**Leads to:** Example 08 (Feature extraction); Unit 3, Unit 4.

---

This notebook covers practical activities from **Course 05, Unit 2**:
- Import/Export using cuDF: Importing and exporting data in different formats using cuDF functions
- Optimization using cuDF: Using GPU acceleration from cuDF to process data faster

---

## Introduction to cuDF

**cuDF** is a GPU-accelerated DataFrame library that provides a pandas-like API for working with data on GPUs. It's part of the RAPIDS ecosystem and can significantly speed up data processing operations.

## The Story

**BEFORE**: You know pandas but don't know how to accelerate data processing with GPU.

**AFTER**: You'll learn cuDF - GPU-accelerated data processing for large datasets (vendors report order-of-magnitude speedups at scale; this notebook measures real timings only when a GPU is present)!

**Why this matters**: cuDF Import/Export and GPU Acceleration is essential for building complete, professional data science solutions!

---

🚀 Google Colab Setup (Run this first if using Colab)


## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- CSV/Parquet files
- cuDF (or pandas fallback)

**Outputs:** What you'll see when you run the cells

- GPU load/save timings
- DataFrames

---

In [1]:
# WHAT: Try to import cuDF; fall back to pandas-only mode when no GPU stack exists.
# WHY: The guard keeps the notebook honest and runnable anywhere - GPU code only runs when a GPU is really present.

# Try importing cuDF (requires CUDA and RAPIDS installation)
try:
    import cudf
    import numpy as np
    import pandas as pd
    CUDF_AVAILABLE = True
    print("✅ cuDF imported successfully!")
    print(f"cuDF version: {cudf.__version__}")
except ImportError:
    CUDF_AVAILABLE = False
    import numpy as np
    import pandas as pd
    print("⚠️  cuDF not available. Install RAPIDS for GPU acceleration:")
    print("   Note: Requires CUDA-capable GPU and RAPIDS installation")
    print("   Continuing with Pandas examples...")

print("✅ Libraries imported!")

⚠️  cuDF not available. Install RAPIDS for GPU acceleration:
   Note: Requires CUDA-capable GPU and RAPIDS installation
   Continuing with Pandas examples...
✅ Libraries imported!


## Part 1: Data Import with cuDF

cuDF supports importing data from CSV, Parquet, JSON, and other formats, similar to Pandas but with GPU acceleration.


In [2]:
# Generate sample data for demonstration
sample_data = pd.DataFrame({
    'id': range(10000), 'value1': np.random.randn(10000),
    'value2': np.random.randint(1, 100, 10000),
    'category': np.random.choice(['A', 'B', 'C'], 10000)
})

# Save to CSV for demonstration
sample_data.to_csv('sample_data_cudf.csv', index=False)
print("✅ Sample CSV file created")

if CUDF_AVAILABLE:
    # Import with cuDF (GPU-accelerated)
    print("\n" + "=" * 60)
    print("Importing with cuDF (GPU-accelerated):")
    print("=" * 60)
    df_cudf = cudf.read_csv('sample_data_cudf.csv')
    print(f"DataFrame shape: {df_cudf.shape}")
    print(f"\nFirst few rows:")
    print(df_cudf.head())
    
    # Import with Pandas (CPU)
    print("\n" + "=" * 60)
    print("Importing with Pandas (CPU):")
    print("=" * 60)
    df_pandas = pd.read_csv('sample_data_cudf.csv')
    print(f"DataFrame shape: {df_pandas.shape}")
else:
    # Demonstrate with Pandas
    print("\n" + "=" * 60)
    print("Importing with Pandas (CPU):")
    print("=" * 60)
    df_pandas = pd.read_csv('sample_data_cudf.csv')
    print(f"DataFrame shape: {df_pandas.shape}")
    print(f"\nFirst few rows:")
    print(df_pandas.head())


✅ Sample CSV file created

Importing with Pandas (CPU):
DataFrame shape: (10000, 4)

First few rows:
   id    value1  value2 category
0   0  1.378095      49        B
1   1  0.077110       8        B
2   2 -0.749485      92        A
3   3 -0.128250       8        C
4   4  0.729762      86        C


## Part 2: Data Export with cuDF

Exporting data to different formats (CSV, Parquet, JSON) using cuDF.


In [3]:
# WHAT: Export the DataFrame to CSV and (with cuDF) to Parquet.
# WHY: Format choice matters at scale - columnar, compressed Parquet is much faster to read back than CSV.

if CUDF_AVAILABLE:
    print("=" * 60)
    print("Exporting with cuDF:")
    print("=" * 60)
    
    # Export to CSV
    df_cudf.to_csv('output_cudf.csv', index=False)
    print("✅ Exported to CSV: output_cudf.csv")
    
    # Export to Parquet (efficient format, good for cuDF)
    df_cudf.to_parquet('output_cudf.parquet')
    print("✅ Exported to Parquet: output_cudf.parquet")
    
    # Note: cuDF also supports JSON, ORC, and other formats
    print("\nNote: cuDF supports CSV, Parquet, JSON, ORC, and other formats")
    print("Parquet format is often preferred for large datasets due to compression and speed")
else:
    # Demonstrate with Pandas
    print("=" * 60)
    print("Exporting with Pandas:")
    print("=" * 60)
    df_pandas.to_csv('output_pandas.csv', index=False)
    print("✅ Exported to CSV: output_pandas.csv")
    print("\nNote: Install cuDF/RAPIDS for GPU-accelerated export and Parquet support")

Exporting with Pandas:
✅ Exported to CSV: output_pandas.csv

Note: Install cuDF/RAPIDS for GPU-accelerated export and Parquet support


## Part 3: GPU Acceleration Performance Comparison

Let's compare cuDF (GPU) vs Pandas (CPU) performance for common operations.

**Honesty note**: on a machine without a CUDA GPU, cuDF cannot run — in that
case the cell below measures the *pandas CPU baseline only* and clearly says
that no GPU numbers were produced here.


In [4]:
# WHAT: Build a 100k-row dataset and time filter/groupby operations (GPU vs CPU when cuDF exists).
# WHY: Measured timings - not vendor claims - show whether the GPU pays off at this data size.

import time

# Create larger dataset for performance comparison
large_data = pd.DataFrame({
    'id': range(100000), 'value1': np.random.randn(100000),
    'value2': np.random.randint(1, 1000, 100000),
    'category': np.random.choice(['A', 'B', 'C', 'D', 'E'], 100000)
})

if CUDF_AVAILABLE:
    # Convert to cuDF
    large_cudf = cudf.from_pandas(large_data)
    
    print("=" * 60)
    print("Performance Comparison: cuDF (GPU) vs Pandas (CPU)")
    print("=" * 60)
    
    # Operation 1: Filtering
    print("\n1. Filtering operations:")
    start = time.time()
    filtered_cudf = large_cudf[large_cudf['value2'] > 500]
    cudf_time = time.time() - start
    print(f"   cuDF (GPU): {cudf_time:.4f} seconds")
    
    start = time.time()
    filtered_pandas = large_data[large_data['value2'] > 500]
    pandas_time = time.time() - start
    print(f"   Pandas (CPU): {pandas_time:.4f} seconds")
    print(f"   Speedup: {pandas_time/cudf_time:.2f}x faster with cuDF")
    
    # Operation 2: Groupby
    print("\n2. Groupby operations:")
    start = time.time()
    grouped_cudf = large_cudf.groupby('category').mean()
    cudf_time = time.time() - start
    print(f"   cuDF (GPU): {cudf_time:.4f} seconds")
    
    start = time.time()
    grouped_pandas = large_data.groupby('category').mean()
    pandas_time = time.time() - start
    print(f"   Pandas (CPU): {pandas_time:.4f} seconds")
    print(f"   Speedup: {pandas_time/cudf_time:.2f}x faster with cuDF")
    
    print("\n✅ GPU acceleration provides significant speedup for large datasets!")
else:
    print("=" * 60)
    print("No CUDA GPU here - measuring the pandas (CPU) baseline only")
    print("=" * 60)

    start = time.time()
    filtered_pandas = large_data[large_data['value2'] > 500]
    pandas_filter_time = time.time() - start
    print(f"\n1. Filtering {len(large_data):,} rows:")
    print(f"   Pandas (CPU): {pandas_filter_time:.4f} seconds → {len(filtered_pandas):,} rows kept")

    start = time.time()
    grouped_pandas = large_data.groupby('category')[['value1', 'value2']].mean()
    pandas_group_time = time.time() - start
    print(f"\n2. Groupby-mean over {large_data['category'].nunique()} categories:")
    print(f"   Pandas (CPU): {pandas_group_time:.4f} seconds")

    print("""
What cuDF would add (NOT measured here - no GPU ran in this notebook):
    - The same filter/groupby API, executed on the GPU
    - NVIDIA's published benchmarks report large speedups on much bigger
      datasets - treat those as the vendor's numbers, not ours
    - Requirements: CUDA-capable NVIDIA GPU + RAPIDS installation

Note: the CPU baseline above is already fast at this size (100k rows).
GPU acceleration matters at much larger scales - tens of millions of rows.""")

No CUDA GPU here - measuring the pandas (CPU) baseline only

1. Filtering 100,000 rows:
   Pandas (CPU): 0.0006 seconds → 50,068 rows kept

2. Groupby-mean over 5 categories:
   Pandas (CPU): 0.0020 seconds

What cuDF would add (NOT measured here - no GPU ran in this notebook):
    - The same filter/groupby API, executed on the GPU
    - NVIDIA's published benchmarks report large speedups on much bigger
      datasets - treat those as the vendor's numbers, not ours
    - Requirements: CUDA-capable NVIDIA GPU + RAPIDS installation

Note: the CPU baseline above is already fast at this size (100k rows).
GPU acceleration matters at much larger scales - tens of millions of rows.


## Summary

### Key Concepts:
1. **cuDF**: GPU-accelerated DataFrame library with pandas-like API
2. **Import/Export**: Supports CSV, Parquet, JSON, ORC formats
3. **GPU Acceleration**: cuDF runs pandas-style operations on the GPU. On this
   machine's run, if no GPU was present, only the pandas CPU baseline was
   measured — the large speedups quoted for cuDF come from NVIDIA's published
   benchmarks on much larger datasets, not from this notebook
4. **Use Cases**: Large-scale data processing, real-time pipelines, accelerated EDA

### Benefits:
- pandas API compatibility → easy migration when a GPU is available
- Integration with the RAPIDS ecosystem
- Worth reaching for when data is far bigger than the 100k rows used here

**Reference:** Course 05, Unit 2: "Import/Export using cuDF" and "Optimization using cuDF: Using GPU acceleration"


## 📚 References

1. McKinney, W. (2010). *Data Structures for Statistical Computing in Python*. Proceedings of the 9th Python in Science Conference (SciPy). <https://doi.org/10.25080/Majora-92bf1922-00a>
2. Raschka, S., Patterson, J., & Nolet, C. (2020). *Machine Learning in Python: Main Developments and Technology Trends in Data Science, Machine Learning, and Artificial Intelligence*. Information, 11(4), 193. <https://arxiv.org/abs/2002.04803>
3. Harris, C. R., Millman, K. J., van der Walt, S. J., et al. (2020). *Array Programming with NumPy*. Nature, 585, 357-362. <https://arxiv.org/abs/2006.10256>